# 05 — Ablation Analyses (Reviewer 2, "not required" comment)

Addresses the three ablation questions raised by Reviewer 2:
1. How many more features can be removed (beyond the top 10) before performance significantly degrades?
2. Which features are redundant with one another?
3. Which features take up the slack if the 50nt-rule feature is removed?

**Note on Q1:** Notebook 03 (Section "Ablation curve: AUC vs number of features") already ran a coarse
version of this at `[1,2,3,4,5,6,7,8,9,10,15,20,30,50,75,100,200,853]` features and found the elbow at
10 features (99% of full-model AUC). That result was only printed/plotted, not saved as data. Section 1
below reruns it at finer granularity from 1-30 features and saves the underlying numbers to CSV so they
can be cited directly in the response-to-reviewers letter and plotted in a supplementary figure.


**Inputs**
- `TOPMed_cleaned.csv` / `config['data']['cleaned']` (from Notebook 02)
- `shap_feature_importance_rankings.csv` (from Notebook 03)

**Outputs** (all under `results/ablation/`)
- `feature_removal_curve.csv` + `feature_removal_curve.png`
- `feature_redundancy_correlation.csv` + `feature_redundancy_heatmap.png` + `redundant_pairs.csv`
- `50nt_rule_ablation_shap_before_after.csv` + `50nt_rule_ablation_summary.json`


## 0. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import json
import yaml
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import shap

# Load config (same conventions as Notebooks 03/04)
config_path = Path("../config/config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

# BASE_DIR anchoring: config.yaml paths (e.g. "data/TOPMed_cleaned.csv") are written relative to the
# repo root (TrunCat/), not to notebooks/. Anchor everything off BASE_DIR so this works regardless of
# where Jupyter's cwd happens to be.
BASE_DIR = config_path.resolve().parent.parent

PATH_INPUT = BASE_DIR / config['data']['cleaned']
TARGET = config['model']['target']
RANDOM_SEED = config['model']['random_seed']
N_FOLDS = config['model']['n_folds']
CATBOOST_PARAMS = config['model']['catboost'].copy()
CATEGORICAL_FEATURES_CONFIG = config['features']['categorical']

FIGURES_DIR = BASE_DIR / Path(config['output']['figures_dir'])
SHAP_RANKINGS_PATH = FIGURES_DIR / "visualizations_cv" / "shap_manuscript" / "shap_feature_importance_rankings.csv"
GENE_IDS_PATH = BASE_DIR / config['data']['gene_ids']

EXPECTED_TREES     = 250
EXPECTED_FOLD_AUCS = [0.7747, 0.7700, 0.8045, 0.7714, 0.7674]

# Toggle to actually write files (per your usual SAVE_OUTPUTS convention)
SAVE_OUTPUTS = True

RESULTS_DIR = BASE_DIR / Path(config['output']['results_dir']) / "ablation"
if SAVE_OUTPUTS:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR:         {BASE_DIR}")
print(f"Input data:       {PATH_INPUT}")
print(f"SHAP rankings:    {SHAP_RANKINGS_PATH}")
print(f"Results dir:      {RESULTS_DIR}")
print(f"Target:           {TARGET}")
print(f"CV folds:         {N_FOLDS}")
print(f"Random seed:      {RANDOM_SEED}")
print(f"SAVE_OUTPUTS:     {SAVE_OUTPUTS}")


In [ ]:
df = pd.read_csv(PATH_INPUT)
y = df[TARGET].astype(int)
drop_cols = [TARGET] + (["key"] if "key" in df.columns else [])
X_full = df.drop(columns=drop_cols)

# Gene groups for StratifiedGroupKFold — same grouping Notebook 03 uses
gene_ids = pd.read_csv(GENE_IDS_PATH)[["key", "GENE_ID"]]
assert "key" in df.columns, "TOPMed_cleaned.csv has no `key` column — stale pre-fix file?"
groups = df[["key"]].merge(gene_ids, on="key", how="left")["GENE_ID"].to_numpy()
assert pd.notna(groups).all(), "some variants have no GENE_ID for grouping"


print(f"\u2713 Loaded cleaned data: {df.shape}")
print(f"  Samples: {len(y)}")
print(f"  Escapees: {y.sum()} ({y.mean()*100:.1f}%)")
print(f"  Features available: {X_full.shape[1]}")

shap_rank_df = pd.read_csv(SHAP_RANKINGS_PATH)
shap_rank_df = shap_rank_df.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
ranked_features = shap_rank_df['feature'].tolist()

assert "key" not in ranked_features and TARGET not in ranked_features, \
    "identifier/target leaked into SHAP rankings"
assert set(ranked_features) <= set(X_full.columns), \
    "SHAP rankings reference columns not in X_full — stale rankings file?"

print(f"\n\u2713 Loaded SHAP rankings: {len(shap_rank_df)} features")
print("\nTop 15 features by mean |SHAP|:")
print(shap_rank_df.head(15).to_string(index=False))

# Full categorical feature list (declared + detected binary/object), matching Notebook 03/04 logic
declared_cat = CATEGORICAL_FEATURES_CONFIG
cat_features_all = [c for c in declared_cat if c in X_full.columns]
other_objs = [c for c in X_full.columns if X_full[c].dtype == "object" and c not in cat_features_all]
cat_features_all.extend(other_objs)
cat_features_all = sorted(set(cat_features_all))
print(f"\nCategorical features detected: {len(cat_features_all)}")

In [ ]:
def train_cv_auc(feature_list, X_full, y, groups, cat_features_all, n_folds, random_seed, catboost_params):
    """Train a 5-fold GENE-GROUPED CV CatBoost model on the given feature subset
    and return (oof_auc, oof_preds, trained_model_on_all_data).

    No early stopping: catboost_params must carry a fixed `iterations` (as the
    corrected TrunCat/TrunKitten training does), since validating against the
    same fold being scored is exactly the leakage bug this notebook is being
    re-run to correct for.
    """
    X = X_full[feature_list].copy()
    cat_sub = [f for f in feature_list if f in cat_features_all]
    for c in cat_sub:
        X[c] = X[c].astype(str).fillna("NA")
    cat_idx = [X.columns.get_loc(c) for c in cat_sub]

    params = catboost_params.copy()
    assert 'iterations' in params, "CATBOOST_PARAMS must specify a fixed iteration count"
    params['random_seed'] = random_seed
    params['verbose'] = False

    skf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=random_seed)
    oof_preds = np.zeros(len(y))

    for tr_idx, va_idx in skf.split(X, y, groups):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        train_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
        valid_pool = Pool(X_va, y_va, cat_features=cat_idx)
        model = CatBoostClassifier(**params)
        model.fit(train_pool)
        oof_preds[va_idx] = model.predict_proba(valid_pool)[:, 1]

    oof_auc = roc_auc_score(y, oof_preds)

    final_pool = Pool(X, y, cat_features=cat_idx)
    final_model = CatBoostClassifier(**params)
    final_model.fit(final_pool)

    return oof_auc, oof_preds, final_model, X, cat_idx

## 1. Feature-Removal Curve (fine-grained, 1-30 features)

Answers: *"How many more features can be removed beyond the top 10 before performance significantly
degrades?"* Reuses the `TOP_N` pattern from Notebook 04. Notebook 03 already showed the elbow sits at
~10 features with performance actually still rising slightly out to ~30 features before flattening; this
reruns every N from 1 to 30 (instead of the coarse `[..,10,15,20,30,..]` steps) so the curve is smooth
enough to cite a precise "can drop N more features" number in the rebuttal.

In [ ]:
FEATURE_COUNTS = list(range(1, 31))  # 1..30 inclusive, fine-grained

removal_curve_results = []

print("="*80)
print("FEATURE-REMOVAL CURVE (1-30 features)")
print("="*80)

for n in FEATURE_COUNTS:
    feats = ranked_features[:n]
    auc, _, _, _, _ = train_cv_auc(feats, X_full, y, groups, cat_features_all, N_FOLDS, RANDOM_SEED, CATBOOST_PARAMS)
    removal_curve_results.append({'n_features': n, 'oof_auc': auc})
    print(f"  N={n:2d} features -> OOF AUC = {auc:.4f}")

removal_curve_df = pd.DataFrame(removal_curve_results)

if SAVE_OUTPUTS:
    removal_curve_df.to_csv(RESULTS_DIR / "feature_removal_curve.csv", index=False)
    print(f"\n\u2713 Saved: {RESULTS_DIR / 'feature_removal_curve.csv'}")

In [ ]:
full_auc_estimate = removal_curve_df['oof_auc'].max()
threshold_99 = full_auc_estimate * 0.99
elbow_row = removal_curve_df[removal_curve_df['oof_auc'] >= threshold_99].iloc[0]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(removal_curve_df['n_features'], removal_curve_df['oof_auc'],
        color='#2E86AB', marker='o', markersize=4, lw=2)
ax.axhline(threshold_99, color='gray', linestyle=':', lw=1.5,
           label=f'99% of best-observed AUC ({threshold_99:.4f})')
ax.axvline(int(elbow_row['n_features']), color='#C73E1D', linestyle='--', lw=1.5,
           label=f"Elbow: {int(elbow_row['n_features'])} features")
ax.set_xlabel('Number of Features (ranked by mean |SHAP|)', fontsize=12, fontweight='bold')
ax.set_ylabel('OOF ROC-AUC', fontsize=12, fontweight='bold')
ax.set_title('Feature-Removal Curve (fine-grained, 1-30 features)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()

if SAVE_OUTPUTS:
    plt.savefig(RESULTS_DIR / "feature_removal_curve.png", dpi=150, bbox_inches='tight')
    print(f"\u2713 Saved: {RESULTS_DIR / 'feature_removal_curve.png'}")
plt.show()

print(f"\nElbow point: {int(elbow_row['n_features'])} features reaches 99% of best-observed OOF AUC ({elbow_row['oof_auc']:.4f})")

## 2. Feature Redundancy Analysis

Answers: *"Which features are redundant with one another?"* Computes pairwise correlation among the
top-30 SHAP features (Pearson for numeric-numeric pairs, correlation ratio for numeric-categorical, and
Cramér's V for categorical-categorical pairs) and flags pairs above a redundancy threshold.

This also directly answers Reviewer 2's separate minor comment: *"Is there a dependence between 3' UTR
features and mRNA half life?"* — check the output table for any `half_life_PC1` row/column.

In [ ]:
TOP_N_REDUNDANCY = 30
REDUNDANCY_THRESHOLD = 0.7  # |correlation| at or above this is flagged as "redundant"

top_features_redundancy = ranked_features[:TOP_N_REDUNDANCY]
X_red = X_full[top_features_redundancy].copy()
cat_red = [f for f in top_features_redundancy if f in cat_features_all]
num_red = [f for f in top_features_redundancy if f not in cat_red]

print(f"Top {TOP_N_REDUNDANCY} features: {len(num_red)} numeric, {len(cat_red)} categorical")
print(f"Categorical: {cat_red}")

In [ ]:
def cramers_v(x, y):
    """Bias-corrected Cramér's V for two categorical series."""
    confusion = pd.crosstab(x, y)
    chi2 = None
    from scipy.stats import chi2_contingency
    chi2 = chi2_contingency(confusion, correction=False)[0]
    n = confusion.sum().sum()
    phi2 = chi2 / n
    r, k = confusion.shape
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    denom = min((kcorr - 1), (rcorr - 1))
    if denom <= 0:
        return 0.0
    return float(np.sqrt(phi2corr / denom))

def correlation_ratio(categorical, numeric):
    """Correlation ratio (eta) between a categorical and a numeric series -- symmetric proxy for
    numeric-categorical association, ranges 0-1."""
    df_tmp = pd.DataFrame({'cat': categorical.astype(str), 'num': numeric})
    df_tmp = df_tmp.dropna()
    if df_tmp['cat'].nunique() < 2 or len(df_tmp) < 2:
        return 0.0
    grand_mean = df_tmp['num'].mean()
    ss_between = df_tmp.groupby('cat')['num'].apply(lambda g: len(g) * (g.mean() - grand_mean) ** 2).sum()
    ss_total = ((df_tmp['num'] - grand_mean) ** 2).sum()
    if ss_total == 0:
        return 0.0
    return float(np.sqrt(ss_between / ss_total))

In [ ]:
n = len(top_features_redundancy)
corr_matrix = pd.DataFrame(np.eye(n), index=top_features_redundancy, columns=top_features_redundancy)

for i in range(n):
    for j in range(i + 1, n):
        f_i, f_j = top_features_redundancy[i], top_features_redundancy[j]
        is_cat_i, is_cat_j = f_i in cat_red, f_j in cat_red
        try:
            if not is_cat_i and not is_cat_j:
                val = X_red[f_i].corr(X_red[f_j], method='pearson')
            elif is_cat_i and is_cat_j:
                val = cramers_v(X_red[f_i].astype(str), X_red[f_j].astype(str))
            elif is_cat_i and not is_cat_j:
                val = correlation_ratio(X_red[f_i], X_red[f_j])
            else:
                val = correlation_ratio(X_red[f_j], X_red[f_i])
        except Exception:
            val = np.nan
        val = 0.0 if pd.isna(val) else val
        corr_matrix.loc[f_i, f_j] = val
        corr_matrix.loc[f_j, f_i] = val

if SAVE_OUTPUTS:
    corr_matrix.to_csv(RESULTS_DIR / "feature_redundancy_correlation.csv")
    print(f"\u2713 Saved: {RESULTS_DIR / 'feature_redundancy_correlation.csv'}")

corr_matrix.iloc[:10, :10].round(2)

In [ ]:
redundant_pairs = []
for i in range(n):
    for j in range(i + 1, n):
        f_i, f_j = top_features_redundancy[i], top_features_redundancy[j]
        val = corr_matrix.loc[f_i, f_j]
        if abs(val) >= REDUNDANCY_THRESHOLD:
            redundant_pairs.append({'feature_a': f_i, 'feature_b': f_j, 'association': round(float(val), 4)})

redundant_pairs_df = pd.DataFrame(redundant_pairs).sort_values('association', ascending=False)
print(f"Found {len(redundant_pairs_df)} pairs with |association| >= {REDUNDANCY_THRESHOLD} among top {TOP_N_REDUNDANCY} features:\n")
print(redundant_pairs_df.to_string(index=False) if len(redundant_pairs_df) else "  (none)")

if SAVE_OUTPUTS:
    redundant_pairs_df.to_csv(RESULTS_DIR / "redundant_pairs.csv", index=False)
    print(f"\n\u2713 Saved: {RESULTS_DIR / 'redundant_pairs.csv'}")

# Specifically surface the 3'UTR vs half_life_PC1 comparisons for Reviewer 2's separate minor comment
if 'half_life_PC1' in top_features_redundancy:
    utr_cols = [c for c in top_features_redundancy if 'utr' in c.lower() or 'UTR' in c]
    if utr_cols:
        print("\n3'UTR feature vs half_life_PC1 associations:")
        print(corr_matrix.loc[utr_cols, 'half_life_PC1'].round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, cbar_kws={'label': 'Association strength'}, ax=ax)
ax.set_title(f'Feature Redundancy Heatmap (top {TOP_N_REDUNDANCY} SHAP features)', fontsize=14, fontweight='bold')
plt.tight_layout()

if SAVE_OUTPUTS:
    plt.savefig(RESULTS_DIR / "feature_redundancy_heatmap.png", dpi=150, bbox_inches='tight')
    print(f"\u2713 Saved: {RESULTS_DIR / 'feature_redundancy_heatmap.png'}")
plt.show()

## 2b. SHAP Interaction Scan (primary redundancy check)

The correlation matrix above answers *"are these features redundant as raw inputs?"* This section
answers the sharper question a reviewer evaluating an ML model actually cares about: *"are these
features redundant in what the model does with them?"*

This computes the full pairwise SHAP interaction tensor from **all 5 CV fold models** (not a single
best-AUC fold) on the same sample of variants, then averages interaction and main-effect magnitudes
across folds -- matching the CV-averaging convention already used for the OOF SHAP values in
`shap_values_for_sharing.pkl` and for the `MedianExpression_log2` x `last.EJC` interaction check in
`SHAP_interaction.ipynb`. This avoids overfitting the redundancy conclusion to whichever single fold
happened to score highest, and the per-fold standard deviation gives a sense of how stable each
interaction is across folds.

**Reading the result table:** low interaction + high correlation (from Section 2) is the strongest
signal of true redundancy — the model isn't combining the features, and they carry overlapping
information, so one could likely be dropped. High interaction (regardless of correlation) means the
features are complementary, not redundant — e.g. your existing `MedianExpression_log2` × `last.EJC`
finding (~83% of main effect) is a complementary pair, not a redundant one.

In [ ]:
import glob

TOP_N_INTERACTION = 20        # feature-set size for the pairwise scan (C(20,2) = 190 pairs)
INTERACTION_SAMPLE = 500      # variants sampled for the interaction-value computation (same sample used across all folds)

CV_MODELS_DIR = BASE_DIR / config['output']['cv_models_dir']

print("Loading all CV fold models...")
fold_models, fold_aucs = [], []
for fold in range(1, N_FOLDS + 1):
    pattern = str(CV_MODELS_DIR / f"fold_{fold}_auc_*.cbm")
    matches = sorted(glob.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No .cbm file found: {pattern}")
    path = matches[0]
    auc_val = float(Path(path).name.split('_auc_')[1].replace('.cbm', ''))
    model = CatBoostClassifier()
    model.load_model(path)
    fold_models.append(model)
    fold_aucs.append(auc_val)
    print(f"  \u2713 Fold {fold}: AUC={auc_val:.4f}")

print(f"\nUsing all {len(fold_models)} fold models (CV-averaged), mean AUC={np.mean(fold_aucs):.4f}")

In [ ]:
# Sample variants once and reuse the SAME sample across all 5 folds (fair average, matching
# SHAP_interaction.ipynb's CV-averaged pattern). Interaction values require the FULL feature matrix
# the model was trained on, not just the top-N subset -- we filter to top-N after computing.
np.random.seed(RANDOM_SEED)
sample_idx = np.random.choice(len(X_full), min(INTERACTION_SAMPLE, len(X_full)), replace=False)
X_sample = X_full.iloc[sample_idx].reset_index(drop=True).copy()

for c in cat_features_all:
    if c in X_sample.columns:
        X_sample[c] = X_sample[c].astype(str).fillna("NA")

print(f"Sampled {len(X_sample)} variants for interaction-value computation")
print(f"Computing SHAP interaction values across all {len(fold_models)} folds (full 853-feature tensor "
      "per fold -- this is the expensive step, budget several minutes per fold)...")

fold_interaction_tensors = []
for fold_idx, model in enumerate(fold_models):
    print(f"  Fold {fold_idx + 1}/{len(fold_models)}...")
    explainer_int = shap.TreeExplainer(model)
    shap_int = explainer_int.shap_interaction_values(X_sample)
    if isinstance(shap_int, list):
        shap_int = shap_int[1]  # escape class
    fold_interaction_tensors.append(shap_int)

# Stack -> shape (n_folds, n_samples, n_features, n_features)
fold_interaction_tensors = np.stack(fold_interaction_tensors, axis=0)
shap_interaction_vals_avg = fold_interaction_tensors.mean(axis=0)   # CV-averaged tensor, used for the scan below
shap_interaction_vals_std = fold_interaction_tensors.std(axis=0)    # per-entry cross-fold stability

print(f"\u2713 CV-averaged interaction tensor shape: {shap_interaction_vals_avg.shape}")

In [ ]:
top_interaction_features = ranked_features[:TOP_N_INTERACTION]
top_indices = [X_sample.columns.get_loc(f) for f in top_interaction_features]

interaction_rows = []
for a in range(len(top_indices)):
    for b in range(a + 1, len(top_indices)):
        f1, f2 = top_interaction_features[a], top_interaction_features[b]
        i1, i2 = top_indices[a], top_indices[b]

        # CV-averaged effects (mean across the 5 fold tensors, computed above)
        interaction_effect = shap_interaction_vals_avg[:, i1, i2]
        main1 = shap_interaction_vals_avg[:, i1, i1]
        main2 = shap_interaction_vals_avg[:, i2, i2]

        # Cross-fold stability: how much does this pair's interaction vary fold-to-fold?
        interaction_fold_std = float(fold_interaction_tensors[:, :, i1, i2].mean(axis=1).std())

        mean_interaction = float(np.abs(interaction_effect).mean())
        mean_main1 = float(np.abs(main1).mean())
        mean_main2 = float(np.abs(main2).mean())
        pct1 = 100 * mean_interaction / mean_main1 if mean_main1 > 0 else 0.0
        pct2 = 100 * mean_interaction / mean_main2 if mean_main2 > 0 else 0.0
        pct_max = max(pct1, pct2)

        if pct_max > 15:
            verdict = 'meaningful interaction (complementary)'
        elif pct_max > 5:
            verdict = 'moderate interaction'
        else:
            verdict = 'weak interaction (near-additive)'

        # Cross-reference with the correlation matrix from Section 2, if this pair is covered there
        corr_val = np.nan
        if f1 in corr_matrix.index and f2 in corr_matrix.columns:
            corr_val = corr_matrix.loc[f1, f2]

        redundancy_flag = (
            'LIKELY REDUNDANT (high corr, low interaction)'
            if (not np.isnan(corr_val) and abs(corr_val) >= REDUNDANCY_THRESHOLD and pct_max < 5)
            else ''
        )

        interaction_rows.append({
            'feature_a': f1, 'feature_b': f2,
            'mean_interaction': mean_interaction,
            'interaction_fold_std': round(interaction_fold_std, 5),
            'pct_of_main_a': round(pct1, 1), 'pct_of_main_b': round(pct2, 1),
            'verdict': verdict,
            'correlation_section2': round(float(corr_val), 3) if not np.isnan(corr_val) else None,
            'redundancy_flag': redundancy_flag,
        })

interaction_scan_df = pd.DataFrame(interaction_rows).sort_values('mean_interaction', ascending=False)

if SAVE_OUTPUTS:
    interaction_scan_df.to_csv(RESULTS_DIR / "shap_interaction_scan.csv", index=False)
    print(f"\u2713 Saved: {RESULTS_DIR / 'shap_interaction_scan.csv'}")

print(f"\nTop 15 strongest interactions among top-{TOP_N_INTERACTION} SHAP features (CV-averaged, n={len(fold_models)} folds):")
print(interaction_scan_df.head(15).to_string(index=False))

redundant_candidates = interaction_scan_df[interaction_scan_df['redundancy_flag'] != '']
print(f"\n{len(redundant_candidates)} pair(s) flagged as likely redundant (high correlation + low interaction):")
print(redundant_candidates.to_string(index=False) if len(redundant_candidates) else "  (none)")

print(f"\nNote: 'interaction_fold_std' shows cross-fold variability -- high std relative to the mean\n"
      f"means this pair's interaction strength isn't stable across folds and should be interpreted cautiously.")

## 2c. SHAP Interaction Scan on Retrained Reduced Models

Section 2b reads interaction values off the **already-trained, full 853-feature TrunCat CV models** --
it shows how the production model uses these features *in the presence of everything else*. That can
understate apparent redundancy: two features might show low pairwise interaction there simply because
some third feature is mediating between them, not because they're independent.

This section instead **retrains fresh 5-fold CV models from scratch** on just a reduced feature subset
(removing the crutch of the other ~830 features), then computes CV-averaged interaction values on
*those* models. This is the more decision-relevant version for a compression question: if two features
still show low interaction once the model has nothing else to lean on, that's much stronger evidence
they're genuinely redundant rather than an artifact of the full feature set.

Set `RETRAIN_FEATURE_SET` below to `'top_n'` (top `TOP_N_RETRAIN` features by SHAP rank -- same set
Section 2b used) or `'trunkitten'` (TrunKitten's actual shipped 10 features, loaded from
`trunkitten_features.json`) to run this against the real deployed reduced model instead of an arbitrary
top-N cut.

In [ ]:
RETRAIN_FEATURE_SET = 'top_n'   # 'top_n' or 'trunkitten'
TOP_N_RETRAIN = 20              # used only when RETRAIN_FEATURE_SET == 'top_n'

# TrunKitten lives in a sibling model directory per the repo's symmetric Model/TrunCat, Model/TrunKitten
# layout -- adjust this path if your repo structure differs.
TRUNKITTEN_FEATURES_PATH = BASE_DIR.parent / "TrunKitten" / "model" / "trunkitten_features.json"

if RETRAIN_FEATURE_SET == 'trunkitten':
    with open(TRUNKITTEN_FEATURES_PATH) as f:
        meta = json.load(f)
    retrain_features = meta["features_in_order"]
    assert len(retrain_features) == meta.get("n_features", len(retrain_features))
    print(f"Using TrunKitten's {len(retrain_features)} shipped features from {TRUNKITTEN_FEATURES_PATH}:")
    print(f"  {retrain_features}")
else:
    retrain_features = ranked_features[:TOP_N_RETRAIN]
    print(f"Using top {TOP_N_RETRAIN} features by SHAP rank:")
    print(f"  {retrain_features}")

In [ ]:
def train_cv_fold_models(feature_list, X_full, y, groups, cat_features_all, n_folds, random_seed, catboost_params):
    """Like train_cv_auc, but returns the 5 individual per-fold models (not just the final
    all-data model) so we can CV-average SHAP interaction values the same way Section 2b does.
    Gene-grouped folds, no early stopping — see train_cv_auc's docstring."""
    X = X_full[feature_list].copy()
    cat_sub = [f for f in feature_list if f in cat_features_all]
    for c in cat_sub:
        X[c] = X[c].astype(str).fillna("NA")
    cat_idx = [X.columns.get_loc(c) for c in cat_sub]

    params = catboost_params.copy()
    assert 'iterations' in params, "CATBOOST_PARAMS must specify a fixed iteration count"
    params['random_seed'] = random_seed
    params['verbose'] = False

    skf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=random_seed)
    oof_preds = np.zeros(len(y))
    fold_models_reduced = []

    for tr_idx, va_idx in skf.split(X, y, groups):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        train_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
        valid_pool = Pool(X_va, y_va, cat_features=cat_idx)
        model = CatBoostClassifier(**params)
        model.fit(train_pool)
        oof_preds[va_idx] = model.predict_proba(valid_pool)[:, 1]
        fold_models_reduced.append(model)

    oof_auc = roc_auc_score(y, oof_preds)
    return oof_auc, fold_models_reduced, X, cat_idx

In [ ]:
print(f"Retraining 5-fold CV models on {len(retrain_features)} features (reduced feature set only)...")
retrain_oof_auc, retrain_fold_models, X_retrain, retrain_cat_idx = train_cv_fold_models(
    retrain_features, X_full, y, groups, cat_features_all, N_FOLDS, RANDOM_SEED, CATBOOST_PARAMS
)
print(f"\u2713 Reduced-model OOF AUC: {retrain_oof_auc:.4f}  (vs. full TrunCat model, see Notebook 03)")

In [ ]:
# Reuse the SAME sampled variant indices as Section 2b for a fair, directly comparable set
X_sample_retrain = X_full.loc[sample_idx, retrain_features].reset_index(drop=True).copy()
for c in retrain_features:
    if c in cat_features_all:
        X_sample_retrain[c] = X_sample_retrain[c].astype(str).fillna("NA")

print(f"Computing SHAP interaction values on the RETRAINED reduced model, across all {len(retrain_fold_models)} folds...")
print(f"(Tensor is only {len(retrain_features)}x{len(retrain_features)} now instead of 853x853 -- much cheaper than Section 2b.)")

retrain_fold_tensors = []
for fold_idx, model in enumerate(retrain_fold_models):
    print(f"  Fold {fold_idx + 1}/{len(retrain_fold_models)}...")
    explainer_r = shap.TreeExplainer(model)
    shap_int_r = explainer_r.shap_interaction_values(X_sample_retrain)
    if isinstance(shap_int_r, list):
        shap_int_r = shap_int_r[1]
    retrain_fold_tensors.append(shap_int_r)

retrain_fold_tensors = np.stack(retrain_fold_tensors, axis=0)
shap_interaction_retrain_avg = retrain_fold_tensors.mean(axis=0)
print(f"\u2713 CV-averaged retrained-model interaction tensor shape: {shap_interaction_retrain_avg.shape}")

In [ ]:
compare_rows = []
n_r = len(retrain_features)
for a in range(n_r):
    for b in range(a + 1, n_r):
        f1, f2 = retrain_features[a], retrain_features[b]

        interaction_r = shap_interaction_retrain_avg[:, a, b]
        main1_r = shap_interaction_retrain_avg[:, a, a]
        main2_r = shap_interaction_retrain_avg[:, b, b]
        mean_int_r = float(np.abs(interaction_r).mean())
        mean_m1_r = float(np.abs(main1_r).mean())
        mean_m2_r = float(np.abs(main2_r).mean())
        pct_r = max(
            100 * mean_int_r / mean_m1_r if mean_m1_r > 0 else 0.0,
            100 * mean_int_r / mean_m2_r if mean_m2_r > 0 else 0.0,
        )

        # Look up the matching full-model (Section 2b) result, if this pair was covered there
        full_model_row = interaction_scan_df[
            ((interaction_scan_df['feature_a'] == f1) & (interaction_scan_df['feature_b'] == f2)) |
            ((interaction_scan_df['feature_a'] == f2) & (interaction_scan_df['feature_b'] == f1))
        ]
        pct_full = float(full_model_row['pct_of_main_a'].combine(full_model_row['pct_of_main_b'], max).iloc[0]) \
            if len(full_model_row) else np.nan

        corr_val = corr_matrix.loc[f1, f2] if (f1 in corr_matrix.index and f2 in corr_matrix.columns) else np.nan

        flag = ''
        if not np.isnan(corr_val) and abs(corr_val) >= REDUNDANCY_THRESHOLD and pct_r < 5:
            flag = 'LIKELY REDUNDANT (confirmed after retraining)'
        elif not np.isnan(pct_full) and pct_r - pct_full > 10:
            flag = 'INTERACTION EMERGED after removing other features'

        compare_rows.append({
            'feature_a': f1, 'feature_b': f2,
            'pct_retrained_model': round(pct_r, 1),
            'pct_full_truncat_model': round(pct_full, 1) if not np.isnan(pct_full) else None,
            'correlation_section2': round(float(corr_val), 3) if not np.isnan(corr_val) else None,
            'flag': flag,
        })

compare_df = pd.DataFrame(compare_rows).sort_values('pct_retrained_model', ascending=False)

if SAVE_OUTPUTS:
    suffix = RETRAIN_FEATURE_SET if RETRAIN_FEATURE_SET == 'trunkitten' else f'top{TOP_N_RETRAIN}'
    compare_df.to_csv(RESULTS_DIR / f"shap_interaction_retrained_vs_full_{suffix}.csv", index=False)
    print(f"\u2713 Saved: {RESULTS_DIR / f'shap_interaction_retrained_vs_full_{suffix}.csv'}")

print(f"\nRetrained-model OOF AUC: {retrain_oof_auc:.4f}\n")
print("Full comparison (retrained-reduced-model interaction % vs. full-TrunCat-model interaction %):")
print(compare_df.to_string(index=False))

flagged = compare_df[compare_df['flag'] != '']
print(f"\n{len(flagged)} pair(s) flagged:")
print(flagged.to_string(index=False) if len(flagged) else "  (none)")

## 3. 50nt-Rule Ablation

Answers: *"Which features take up the slack if the 50nt-rule feature is removed?"* Drops the feature
operationalizing the canonical PTC-to-last-EJC distance rule, retrains on the same feature set minus that
one feature, and compares SHAP importance rankings before vs. after to see which features absorb its
signal.

**Update `RULE_50NT_FEATURE` below if `last.EJC` isn't the right column for your definition of the
"50nt rule."**

In [ ]:
RULE_50NT_FEATURE = "last.EJC"
ABLATION_TOP_N = 30  # feature-set size to retrain on (with and without the 50nt-rule feature)

assert RULE_50NT_FEATURE in ranked_features, (
    f"'{RULE_50NT_FEATURE}' not found in SHAP rankings -- update RULE_50NT_FEATURE to the correct column name."
)

baseline_features = ranked_features[:ABLATION_TOP_N]
ablated_features = [f for f in baseline_features if f != RULE_50NT_FEATURE]
# Backfill with the next-ranked feature so both models see the same *number* of features
backfill_candidates = [f for f in ranked_features if f not in baseline_features]
if backfill_candidates:
    ablated_features.append(backfill_candidates[0])

print(f"Baseline feature set (n={len(baseline_features)}): includes '{RULE_50NT_FEATURE}'")
print(f"Ablated feature set  (n={len(ablated_features)}): '{RULE_50NT_FEATURE}' removed"
      + (f", backfilled with '{backfill_candidates[0]}'" if backfill_candidates else ""))

In [ ]:
print("Training BASELINE model (with 50nt-rule feature)...")
baseline_auc, baseline_oof, baseline_model, X_baseline, baseline_cat_idx = train_cv_auc(
    baseline_features, X_full, y, groups, cat_features_all, N_FOLDS, RANDOM_SEED, CATBOOST_PARAMS
)
print(f"  Baseline OOF AUC: {baseline_auc:.4f}")

print("\nTraining ABLATED model (50nt-rule feature removed)...")
ablated_auc, ablated_oof, ablated_model, X_ablated, ablated_cat_idx = train_cv_auc(
    ablated_features, X_full, y, groups, cat_features_all, N_FOLDS, RANDOM_SEED, CATBOOST_PARAMS
)
print(f"  Ablated OOF AUC:  {ablated_auc:.4f}")
print(f"  \u0394 AUC:           {ablated_auc - baseline_auc:+.4f}")

In [ ]:
def mean_abs_shap(model, X, cat_idx):
    pool = Pool(X, cat_features=cat_idx)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(pool)
    return pd.DataFrame({
        'feature': X.columns,
        'mean_abs_shap': np.abs(shap_values).mean(axis=0)
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print("Computing SHAP for baseline model...")
shap_before = mean_abs_shap(baseline_model, X_baseline, baseline_cat_idx)
shap_before['rank'] = np.arange(1, len(shap_before) + 1)

print("Computing SHAP for ablated model...")
shap_after = mean_abs_shap(ablated_model, X_ablated, ablated_cat_idx)
shap_after['rank'] = np.arange(1, len(shap_after) + 1)

comparison = shap_before[['feature', 'mean_abs_shap', 'rank']].merge(
    shap_after[['feature', 'mean_abs_shap', 'rank']],
    on='feature', how='outer', suffixes=('_before', '_after')
)
comparison['rank_before'] = comparison['rank_before'].fillna(len(shap_before) + 1).astype(int)
comparison['rank_after'] = comparison['rank_after'].fillna(len(shap_after) + 1).astype(int)
comparison['rank_change'] = comparison['rank_before'] - comparison['rank_after']  # positive = moved up (more important) after ablation
comparison = comparison.sort_values('rank_change', ascending=False)

if SAVE_OUTPUTS:
    comparison.to_csv(RESULTS_DIR / "50nt_rule_ablation_shap_before_after.csv", index=False)
    print(f"\u2713 Saved: {RESULTS_DIR / '50nt_rule_ablation_shap_before_after.csv'}")

print(f"\nFeatures that gained the most SHAP importance after removing '{RULE_50NT_FEATURE}':")
print(comparison.head(10).to_string(index=False))

In [ ]:
top_absorbers = comparison[comparison['feature'] != RULE_50NT_FEATURE].sort_values('rank_change', ascending=False).head(5)

summary = {
    'rule_50nt_feature': RULE_50NT_FEATURE,
    'ablation_top_n': ABLATION_TOP_N,
    'baseline_oof_auc': float(baseline_auc),
    'ablated_oof_auc': float(ablated_auc),
    'delta_auc': float(ablated_auc - baseline_auc),
    'top_absorbing_features': top_absorbers[['feature', 'rank_before', 'rank_after', 'rank_change']].to_dict('records'),
}

if SAVE_OUTPUTS:
    with open(RESULTS_DIR / "50nt_rule_ablation_summary.json", 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    print(f"\u2713 Saved: {RESULTS_DIR / '50nt_rule_ablation_summary.json'}")

print(json.dumps(summary, indent=2, default=str))

## Done

Summary for the response-to-reviewers letter:
- **Q1 (feature-removal curve):** see `feature_removal_curve.csv` / `.png` for the elbow point and exact AUC at each N.
- **Q2 (redundancy):** three complementary views --
  - `redundant_pairs.csv`: raw correlation among top features (cheap, naive-input-level redundancy)
  - `shap_interaction_scan.csv`: interaction values from the full 853-feature TrunCat model (how the production model actually uses these features in context)
  - `shap_interaction_retrained_vs_full_*.csv`: interaction values after retraining on just the reduced feature set (removes the crutch of other features -- the most decision-relevant view for compression questions, and can be run directly against TrunKitten's real 10 features)
  
  Pairs flagged `LIKELY REDUNDANT` in the retrained comparison are the strongest redundancy candidates. Also answers R2's separate 3'UTR-vs-half-life comment.
- **Q3 (50nt-rule ablation):** see `50nt_rule_ablation_summary.json` for the AUC delta and which features absorbed the removed feature's signal.

In [ ]:
# ============================================================================
# Section 2d — TrunKitten swap test: does removing correlated pairs help/hurt?
# (Paste after Section 2c has been run at least once, so the standard
# setup cells -- ranked_features, X_full, y, groups, cat_features_all, N_FOLDS,
# RANDOM_SEED, CATBOOST_PARAMS, RESULTS_DIR, SAVE_OUTPUTS -- are in memory.)
# ============================================================================

# --- The actual shipped TrunKitten feature set (corrected top-10 under
# StratifiedGroupKFold; cdsseq_AUcontentlast200 fell out of the top 10
# entirely and MedianExpression_log2 took rank 10) ---
TRUNKITTEN_FEATURES = [
    'last.EJC', 'relativePTClocation', 'half_life_PC1', 'cdsseqs_AU_content',
    'mut.exon', 'phastcons_new3utr_first200_median', 'phylop_ptc_to_ejc_median',
    'AmountExonsAfter', 'cdsseqs_UC_content', 'MedianExpression_log2',
]

# The two pairs flagged as redundant/borderline within this feature set
REDUNDANT_CLUSTER_FEATURES = {
    'phastcons_new3utr_first200_median', 'phylop_new3utr_first200_median',
    'phastcons_ptc_to_ejc_median', 'phylop_ptc_to_ejc_median',
    'cdsseqs_AU_content', 'cdsseq_AUcontentlast200',
}

# Next-best independent features: top of the SHAP ranking, excluding anything
# already in TrunKitten AND anything in the redundant cluster (so we're testing
# a genuinely independent replacement, not swapping one redundant feature for another)
candidates = [f for f in ranked_features
              if f not in TRUNKITTEN_FEATURES and f not in REDUNDANT_CLUSTER_FEATURES]
replacement_1, replacement_2 = candidates[0], candidates[1]

print(f"Replacement candidates (next-best independent features): {replacement_1}, {replacement_2}")

# --- Build the swap variants ---
# Targets are the two LOFO-unstable rank-9/10 features (cdsseqs_UC_content:
# 3/5 folds, MedianExpression_log2: 4/5 folds), not cdsseq_AUcontentlast200,
# which isn't in the corrected top 10 at all.
swap_UC = [f for f in TRUNKITTEN_FEATURES if f != 'cdsseqs_UC_content'] + [replacement_1]
swap_expr = [f for f in TRUNKITTEN_FEATURES if f != 'MedianExpression_log2'] + [replacement_2]
swap_both = ([f for f in TRUNKITTEN_FEATURES if f not in ('cdsseqs_UC_content', 'MedianExpression_log2')]
             + [replacement_1, replacement_2])

variants = {
    'trunkitten_10_baseline': TRUNKITTEN_FEATURES,
    f'swap_UCcontent_for_{replacement_1}': swap_UC,
    f'swap_expression_for_{replacement_2}': swap_expr,
    'swap_both': swap_both,
}

for name, feats in variants.items():
    print(f"\n{name}: {feats}")

In [ ]:
# --- Train all four variants and compare ---
swap_results = []

for name, feats in variants.items():
    print(f"\nTraining '{name}' ({len(feats)} features)...")
    auc, _, _, _ = train_cv_fold_models(feats, X_full, y, groups, cat_features_all, N_FOLDS, RANDOM_SEED, CATBOOST_PARAMS)
    print(f"  OOF AUC: {auc:.4f}")
    swap_results.append({'variant': name, 'features': feats, 'oof_auc': auc})

swap_results_df = pd.DataFrame(swap_results)
baseline_auc_swap = swap_results_df.loc[swap_results_df['variant'] == 'trunkitten_10_baseline', 'oof_auc'].iloc[0]
swap_results_df['delta_vs_current'] = swap_results_df['oof_auc'] - baseline_auc_swap

if SAVE_OUTPUTS:
    swap_results_df.to_csv(RESULTS_DIR / "trunkitten_redundancy_swap_test.csv", index=False)
    print(f"\n\u2713 Saved: {RESULTS_DIR / 'trunkitten_redundancy_swap_test.csv'}")

print("\n" + "="*80)
print("TOP10 SWAP TEST RESULTS")
print("="*80)
print(swap_results_df[['variant', 'oof_auc', 'delta_vs_current']].to_string(index=False))

print(f"\nTop 10 (with redundant pairs): OOF AUC = {baseline_auc_swap:.4f}")
best_row = swap_results_df.loc[swap_results_df['oof_auc'].idxmax()]
print(f"Best variant: {best_row['variant']}  (OOF AUC = {best_row['oof_auc']:.4f}, "
      f"\u0394 = {best_row['delta_vs_current']:+.4f})")

In [ ]:
# ============================================================================
# Section 2e — Drop-without-replace test: does going smaller than 10 help?
# Tests whether simply removing the two LOFO-unstable features (no
# replacement, N=8 -- the actual shipped set) costs meaningful performance
# vs. the swap variants and the current 10.
# ============================================================================

TRUNKITTEN_MINUS_REDUNDANT = [f for f in TRUNKITTEN_FEATURES
                              if f not in ('cdsseqs_UC_content', 'MedianExpression_log2')]
# This is the shipped 8.

TRUNKITTEN_MINUS_UC_ONLY = [f for f in TRUNKITTEN_FEATURES if f != 'cdsseqs_UC_content']
TRUNKITTEN_MINUS_EXPRESSION_ONLY = [f for f in TRUNKITTEN_FEATURES if f != 'MedianExpression_log2']

drop_variants = {
    'drop_UCcontent_only_N9': TRUNKITTEN_MINUS_UC_ONLY,
    'drop_expression_only_N9': TRUNKITTEN_MINUS_EXPRESSION_ONLY,
    'drop_both_N8': TRUNKITTEN_MINUS_REDUNDANT,
}

drop_results = []
for name, feats in drop_variants.items():
    print(f"Training '{name}' ({len(feats)} features)...")
    auc, _, _, _ = train_cv_fold_models(feats, X_full, y, groups, cat_features_all, N_FOLDS, RANDOM_SEED, CATBOOST_PARAMS)
    print(f"  OOF AUC: {auc:.4f}")
    drop_results.append({'variant': name, 'n_features': len(feats), 'features': feats, 'oof_auc': auc})

drop_results_df = pd.DataFrame(drop_results)
drop_results_df['delta_vs_current_10'] = drop_results_df['oof_auc'] - baseline_auc_swap

if SAVE_OUTPUTS:
    drop_results_df.to_csv(RESULTS_DIR / "trunkitten_drop_no_replace_test.csv", index=False)
    print(f"\n✓ Saved: {RESULTS_DIR / 'trunkitten_drop_no_replace_test.csv'}")

combined = pd.concat([
    swap_results_df[['variant', 'oof_auc', 'delta_vs_current']].rename(columns={'delta_vs_current': 'delta_vs_current_10'}),
    drop_results_df[['variant', 'oof_auc', 'delta_vs_current_10']],
], ignore_index=True)

print("\n" + "="*80)
print("FULL COMPARISON: keep-10 vs. swap vs. drop-without-replace")
print("="*80)
print(combined.to_string(index=False))

In [ ]:
# ============================================================================
# Section 4 — Response Figure 1 (multi-panel summary for the rebuttal letter)
# Reads from RESULTS_DIR so this works even in a fresh kernel, as long as
# Sections 1-3 and the TrunKitten swap-test cells have been run at least once.
# ============================================================================

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np

removal_df = pd.read_csv(RESULTS_DIR / "feature_removal_curve.csv")
corr_df = pd.read_csv(RESULTS_DIR / "feature_redundancy_correlation.csv", index_col=0)
ablation_df = pd.read_csv(RESULTS_DIR / "50nt_rule_ablation_shap_before_after.csv")
swap_df = pd.read_csv(RESULTS_DIR / "trunkitten_redundancy_swap_test.csv")

def short_label(name, max_len=22):
    """Truncate long feature names for tick labels, keeping them unique-ish."""
    return name if len(name) <= max_len else name[:max_len - 1] + "\u2026"

fig = plt.figure(figsize=(16, 14))  # taller figure for more breathing room
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.50, wspace=0.35)  # more hspace

# ---- Panel A: Feature-removal curve ----
ax_a = fig.add_subplot(gs[0, 0])
full_auc = removal_df['oof_auc'].max()
thresh99 = full_auc * 0.99
elbow_n = removal_df.loc[removal_df['oof_auc'] >= thresh99, 'n_features'].iloc[0]
ax_a.plot(removal_df['n_features'], removal_df['oof_auc'], color='#2E86AB', marker='o', markersize=4, lw=2)
ax_a.axhline(thresh99, color='gray', linestyle=':', lw=1.2, label=f'99% of best AUC ({thresh99:.3f})')
ax_a.axvline(elbow_n, color='#C73E1D', linestyle='--', lw=1.2, label=f'Elbow: {int(elbow_n)} features')
ax_a.axvline(10, color='#2A9D8F', linestyle='-.', lw=1.2, label='TrunKitten (N=10)')
ax_a.set_xlabel('Number of Features (ranked by mean |SHAP|)', fontweight='bold')
ax_a.set_ylabel('OOF ROC-AUC', fontweight='bold')
ax_a.set_title('A. Feature-Removal Curve', fontweight='bold', fontsize=13, loc='left')
ax_a.legend(fontsize=8, loc='lower right')
ax_a.grid(alpha=0.3)
ax_a.spines[['top', 'right']].set_visible(False)

# ---- Panel B: Redundancy heatmap (top 12 for readability, truncated labels) ----
ax_b = fig.add_subplot(gs[0, 1])
TOP_B = 12  # reduced from 15 -- fewer, cleaner labels
top_b = corr_df.iloc[:TOP_B, :TOP_B]
b_labels = [short_label(f) for f in top_b.columns]
im = ax_b.imshow(top_b.values, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax_b.set_xticks(range(len(b_labels)))
ax_b.set_yticks(range(len(b_labels)))
ax_b.set_xticklabels(b_labels, rotation=45, ha='right', fontsize=7)
ax_b.set_yticklabels(b_labels, fontsize=7)
ax_b.set_title('B. Feature Redundancy (top 12 by SHAP)', fontweight='bold', fontsize=13, loc='left', pad=10)
cbar = fig.colorbar(im, ax=ax_b, fraction=0.046, pad=0.04)
cbar.set_label('Association strength', fontsize=8)
cbar.ax.tick_params(labelsize=7)

# ---- Panel C: 50nt-rule ablation compensation ----
ax_c = fig.add_subplot(gs[1, 0])
ablation_df['shap_delta'] = ablation_df['mean_abs_shap_after'] - ablation_df['mean_abs_shap_before']
plot_df = ablation_df.dropna(subset=['shap_delta']).copy()
plot_df = plot_df.reindex(plot_df['shap_delta'].abs().sort_values(ascending=False).index).head(10)
plot_df = plot_df.sort_values('shap_delta')
plot_df['label'] = plot_df['feature'].apply(short_label)
colors = ['#C73E1D' if v < 0 else '#2A9D8F' for v in plot_df['shap_delta']]
ax_c.barh(plot_df['label'], plot_df['shap_delta'], color=colors)
ax_c.axvline(0, color='black', lw=0.8)
ax_c.set_xlabel('\u0394 mean |SHAP| (after \u2212 before removing last.EJC)', fontweight='bold', fontsize=9)
ax_c.set_title('C. 50nt-Rule Ablation: Compensating Features', fontweight='bold', fontsize=13, loc='left', pad=10)
ax_c.tick_params(axis='y', labelsize=8)
ax_c.spines[['top', 'right']].set_visible(False)

# ---- Panel D: TrunKitten redundancy swap test ----
ax_d = fig.add_subplot(gs[1, 1])
label_map = {'trunkitten_10_baseline': 'Ten baseline\nTrunKitten'}
labels = [label_map.get(v, v.replace('_', '\n')) for v in swap_df['variant']]
bar_colors = ['#6C757D' if v == 'trunkitten_10_baseline' else '#2A9D8F' for v in swap_df['variant']]
bars = ax_d.bar(labels, swap_df['oof_auc'], color=bar_colors)
baseline = swap_df.loc[swap_df['variant'] == 'trunkitten_10_baseline', 'oof_auc'].iloc[0]
ax_d.axhline(baseline, color='gray', linestyle=':', lw=1)
for bar, delta in zip(bars, swap_df['delta_vs_current']):
    label = f"+{delta:.4f}" if delta > 0 else f"{delta:.4f}"
    ax_d.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.0005,
               label, ha='center', fontsize=8, fontweight='bold')
ax_d.set_ylim(baseline - 0.005, swap_df['oof_auc'].max() + 0.003)
ax_d.set_ylabel('OOF ROC-AUC', fontweight='bold')
ax_d.set_title('D. TrunKitten Redundancy Swap Test', fontweight='bold', fontsize=13, loc='left', pad=10)
ax_d.tick_params(axis='x', labelsize=8)
ax_d.spines[['top', 'right']].set_visible(False)

fig.suptitle('Response Figure 1: Ablation, Redundancy, and Compensation Analyses',
             fontsize=15, fontweight='bold', y=1.00)

plt.tight_layout()

if SAVE_OUTPUTS:
    fig.savefig(RESULTS_DIR / "response_figure_1_composite.png", dpi=200, bbox_inches='tight')
    print(f"\u2713 Saved: {RESULTS_DIR / 'response_figure_1_composite.png'}")

plt.show()

In [ ]:
# ============================================================================
# Response Figure 2 — Full TrunKitten variant comparison (swap vs. drop)
# ============================================================================

swap_df = pd.read_csv(RESULTS_DIR / "trunkitten_redundancy_swap_test.csv")
drop_df = pd.read_csv(RESULTS_DIR / "trunkitten_drop_no_replace_test.csv")

# Build a unified table with feature count and category for each variant
rows = []
for _, r in swap_df.iterrows():
    n = 10  # all swap variants keep N=10
    cat = 'current' if r['variant'] == 'trunkitten_10_baseline' else 'swap (N=10)'
    rows.append({'variant': r['variant'], 'n_features': n, 'oof_auc': r['oof_auc'], 'category': cat})
for _, r in drop_df.iterrows():
    rows.append({'variant': r['variant'], 'n_features': r['n_features'], 'oof_auc': r['oof_auc'],
                 'category': f"drop (N={r['n_features']})"})

all_variants = pd.DataFrame(rows).sort_values('oof_auc').reset_index(drop=True)

nice_labels = {
    'trunkitten_10_baseline': 'Baseline 10\nTrunKitten (N=10)',
    'drop_expression_only_N9': 'Drop expression\n(N=9)',
    'drop_both_N8': 'Drop both\n(N=8, shipped)',
    'drop_UCcontent_only_N9': 'Drop UC-content\n(N=9)',
    f'swap_UCcontent_for_{replacement_1}': 'Swap UC-content\n(N=10)',
    f'swap_expression_for_{replacement_2}': 'Swap expression\n(N=10)',
    'swap_both': 'Swap both\n(N=10)',
}
all_variants['label'] = all_variants['variant'].map(lambda v: nice_labels.get(v, v))

color_map = {'current': '#6C757D', 'drop (N=9)': '#E9A03B', 'drop (N=8)': '#C73E1D', 'swap (N=10)': '#2A9D8F'}
bar_colors = all_variants['category'].map(color_map)

fig2, ax2 = plt.subplots(figsize=(11, 6))
bars = ax2.bar(all_variants['label'], all_variants['oof_auc'], color=bar_colors)

baseline = all_variants.loc[all_variants['category'] == 'current', 'oof_auc'].iloc[0]
ax2.axhline(baseline, color='gray', linestyle=':', lw=1)

for bar, auc in zip(bars, all_variants['oof_auc']):
    delta = auc - baseline
    label = f"{auc:.4f}\n({'+' if delta > 0 else ''}{delta:.4f})" if delta != 0 else f"{auc:.4f}\n(baseline)"
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.0003,
              label, ha='center', fontsize=8, fontweight='bold')

ax2.set_ylim(baseline - 0.004, all_variants['oof_auc'].max() + 0.004)
ax2.set_ylabel('OOF ROC-AUC', fontweight='bold')
ax2.set_title('Response Figure 2: TrunKitten Feature-Count Trade-off\n(Swapping redundant features outperforms simply dropping them)',
              fontweight='bold', fontsize=12)
ax2.tick_params(axis='x', labelsize=8)
ax2.spines[['top', 'right']].set_visible(False)

# Legend for the color categories
from matplotlib.patches import Patch
legend_elems = [Patch(facecolor=color_map['current'], label='Current (N=10)'),
                Patch(facecolor=color_map['drop (N=9)'], label='Drop, no replace (N=9)'),
                Patch(facecolor=color_map['drop (N=8)'], label='Drop both, no replace (N=8)'),
                Patch(facecolor=color_map['swap (N=10)'], label='Swap for independent feature (N=10)')]
ax2.legend(handles=legend_elems, fontsize=8, loc='upper left')

plt.tight_layout()

if SAVE_OUTPUTS:
    fig2.savefig(RESULTS_DIR / "response_figure_2_dropvsswap.png", dpi=200, bbox_inches='tight')
    print(f"\u2713 Saved: {RESULTS_DIR / 'response_figure_2_dropvsswap.png'}")

plt.show()